# News Summarization Eval

Colab-friendly notebook for running the news summarization benchmark on a strong open-weight Hugging Face model.

This notebook pulls the benchmark code directly from the Hugging Face repo so it stays in sync with the HF page.

Recommended first model:
- `Qwen/Qwen2.5-3B-Instruct`


In [ ]:
!pip install -q transformers accelerate sentencepiece bert-score rouge-score huggingface_hub

In [ ]:
from google.colab import userdata
from huggingface_hub import login
import os

token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = token
login(token=token)
print("HF token loaded from Colab Secrets")


In [ ]:
from huggingface_hub import snapshot_download

snapshot_download(
    repo_id="arach/training-lab",
    repo_type="model",
    local_dir="/content/training-lab-hf",
    allow_patterns=[
        "eval/news_summarization/*",
    ],
)

%cd /content/training-lab-hf

In [ ]:
MODEL = "Qwen/Qwen2.5-3B-Instruct"
LIMIT = 10
PROMPT_STYLE = "simple"
DTYPE = "bfloat16"
TRUST_REMOTE_CODE = False
DISABLE_BERTSCORE = True

In [ ]:
trust_flag = "--trust-remote-code" if TRUST_REMOTE_CODE else ""
bertscore_flag = "--disable-bertscore" if DISABLE_BERTSCORE else ""
!python eval/news_summarization/run_hf_transformers.py \
  --model {MODEL} \
  --limit {LIMIT} \
  --prompt-style {PROMPT_STYLE} \
  --dtype {DTYPE} \
  --verbose \
  {bertscore_flag} \
  {trust_flag}

In [ ]:
trust_flag = "--trust-remote-code" if TRUST_REMOTE_CODE else ""
bertscore_flag = "--disable-bertscore" if DISABLE_BERTSCORE else ""
!python eval/news_summarization/run_hf_transformers.py \
  --model {MODEL} \
  --limit {LIMIT} \
  --prompt-style {PROMPT_STYLE} \
  --dtype {DTYPE} \
  --resume \
  {bertscore_flag} \
  {trust_flag}

In [ ]:
import json
from pathlib import Path

results_dir = Path('eval/news_summarization/results')
results_dir.mkdir(parents=True, exist_ok=True)
files = sorted(results_dir.glob('hf-transformers-*.json'))
print('Results dir:', results_dir)
for path in files:
    print(path.name, f'({path.stat().st_size} bytes)')

if not files:
    print('No result file yet')
else:
    latest = files[-1]
    print('\nLatest:', latest)
    data = json.loads(latest.read_text())
    print(json.dumps(data.get('summary', {}), indent=2))
